<font size=10>**TARGET & FEATURE REPRESENTATIONS**</font> <a class="anchor" id='title'></a>

**Bachelor's in Data Science - NOVA IMS (26/27)**

**Course**: *Data Stream Mining — Practical Assignment I (PA48)*

**Project**: *Weather Windows for Outdoor Races — Angra do Heroísmo vs Braga*

**Group YY**
- Beatriz Marques 20231605
- Duarte Fernandes 20231619

**Question**: *Which city offers more reliable weather windows for recurring outdoor races?*

**Pipeline step**: This notebook reproduces **step 2** and the target/feature/mask part of **step 3** of `python src/practical_assignment.py --config config.yaml`: freeze the pooled warm-up threshold, build T6, construct representations A and B, and use the same eligibility mask for both representations.

| Notebook | What it does |
|---|---|
| 01 — `01_data_collection_and_quality.ipynb` | Data collection and quality |
| **02 — `02_target_and_features.ipynb`** | **Target, features and shared mask** |
| 03 — `03_prequential_evaluation.ipynb` | Evaluation and audit outputs |
| 04 — `04_results_and_figures.ipynb` | Results, figures and audit checks |

<font color='#BFD72F' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>

- [1. Imports](#1)

- [2. Data](#2)

- [3. Target T6 — Adverse Outdoor Event](#3)
    - [3.1 Frozen Threshold from the Pooled Warm-up](#3_1)
    - [3.2 Building $y_t$](#3_2)
    - [3.3 Class Balance](#3_3)
        - [Insights](#insights_1)

- [4. Feature Representations](#4)
    - [4.1 Representation A — Previous State](#4_1)
    - [4.2 Representation B — Stream-aware Context](#4_2)
    - [4.3 Leakage Checks](#4_3)
    - [4.4 Shared Evaluation Mask](#4_4)
        - [Insights](#insights_2)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

Enable **automatic source reloads** and suppress warnings while running the notebook cells in order. Start Jupyter with this notebook in the repository’s `notebooks/` folder.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

Import the libraries and the documented **`src` API**. Add `../src` to `sys.path`; subsequent data and output paths are built from **`data_loader.ROOT`**, not the notebook working directory.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Get the absolute path of the source code folder
source_code_path = os.path.abspath('../src')

# Add the source code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

from data_loader import ROOT, load_config, fetch_city
from targets import RAIN_MM, compute_thresholds, make_target
from features import BASE, representation_a, representation_b

# <font color='#BFD72F' size=6>**2. Data**</font> <a class="anchor" id="2"></a>

[Back to TOC](#toc)

Reload the configuration and fetch both cities through **`fetch_city`** using its default cache. Keep **`data`** and **`names`** keyed by `city_a` / `city_b`, as in the terminal pipeline. The raw path is resolved from `ROOT`.

In [ ]:
cfg = load_config(ROOT / 'config.yaml')
period = cfg['period']
warm_n = cfg['warmup']['n_hours']
raw_dir = ROOT / cfg['paths']['raw_data']
data = {}
names = {}
for key in ('city_a', 'city_b'):
    c = cfg['cities'][key]
    names[key] = c['name']
    data[key] = fetch_city(
        c['name'], c['latitude'], c['longitude'],
        period['start_date'], period['end_date'], period['timezone'],
        raw_dir=raw_dir,
    )
target = cfg['assignment']['target']
assert target == 'T6', 'This notebook is designed for the configured T6 assignment.'
display(pd.DataFrame([{'city': names[k], 'rows': len(v)} for k, v in data.items()]))

# <font color='#BFD72F' size=6>**3. Target T6 — Adverse Outdoor Event**</font> <a class="anchor" id="3"></a>

[Back to TOC](#toc)

The **T6** label is built by `make_target` using the frozen wind threshold and `RAIN_MM = 0.1`:

$y_t = 1 \iff P_t \ge 0.1\ \text{mm} \lor W_t \ge \tau_W$

Here, $P_t$ is precipitation and $W_t$ is wind speed at time $t$. The current measurements define the **label**, not the current feature vector. `make_target` leaves the label missing where its source data are missing.

## <font color='#BFD72F' size=5>3.1 Frozen Threshold from the Pooled Warm-up</font> <a class="anchor" id="3_1"></a>

[Back to TOC](#toc)

Call **`compute_thresholds`** with the two complete streams, **`cfg["warmup"]["n_hours"]`** and **`cfg["assignment"]["target"]`**. For T6, `thresholds["T5"]` is the pooled warm-up wind **0.80 quantile**. Do not recompute it on post-warm-up data.

In [ ]:
thresholds = compute_thresholds(data['city_a'], data['city_b'], warm_n, target)
tau_W = thresholds['T5']
print(f'Frozen tau_W (pooled warm-up wind Q0.80): {tau_W}')

## <font color='#BFD72F' size=5>3.2 Building $y_t$</font> <a class="anchor" id="3_2"></a>

[Back to TOC](#toc)

Build one **`y`** Series per city with `make_target(df, target, thresholds)`. Keep its original index and missing values; filtering is deferred to the shared mask.

In [ ]:
y = {key: make_target(df, target, thresholds) for key, df in data.items()}
display(pd.concat({names[key]: series for key, series in y.items()}, axis=1).head())

## <font color='#BFD72F' size=5>3.3 Class Balance</font> <a class="anchor" id="3_3"></a>

[Back to TOC](#toc)

Calculate event rates over **non-missing labels** in all, warm-up and post-warm-up rows, before feature filtering. Decompose **labelled events** into rain only, wind only and both; their denominator is the number of labelled events in the full stream. Empty denominators are shown as `NaN`.

In [ ]:
balance_rows = []
for key, df in data.items():
    labels_y = y[key]
    warm = pd.Series(df.index < warm_n, index=df.index)
    rain = df['precipitation'].ge(RAIN_MM)
    wind = df['wind_speed'].ge(tau_W)
    events = labels_y.eq(1) & labels_y.notna()
    n_events = int(events.sum())
    shares = {
        'rain_only_share': int((events & rain & ~wind).sum()),
        'wind_only_share': int((events & wind & ~rain).sum()),
        'both_share': int((events & rain & wind).sum()),
    }
    balance_rows.append({
        'city': names[key], 'labelled_rows': int(labels_y.notna().sum()),
        'event_rate_all': labels_y.mean(),
        'event_rate_warmup': labels_y[warm].mean(),
        'event_rate_post_warmup': labels_y[~warm].mean(),
        **{column: count / n_events if n_events else np.nan for column, count in shares.items()},
    })
balance_df = pd.DataFrame(balance_rows).set_index('city')
display(balance_df)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
balance_df[['event_rate_all', 'event_rate_warmup', 'event_rate_post_warmup']].plot.bar(ax=axes[0])
axes[0].set_title('T6 event rates by stream partition')
axes[0].set_ylabel('Share of non-missing labels')
axes[0].set_xlabel('City')
axes[0].set_ylim(0, 1)
balance_df[['rain_only_share', 'wind_only_share', 'both_share']].plot.bar(stacked=True, ax=axes[1])
axes[1].set_title('T6 events by source condition')
axes[1].set_ylabel('Share of labelled events')
axes[1].set_xlabel('City')
axes[1].set_ylim(0, 1)
for ax in axes:
    ax.tick_params(axis='x', rotation=0)
fig.tight_layout()
plt.show()

#### **Insights** <a class="anchor" id='insights_1'></a>
*To be written by the group.*

# <font color='#BFD72F' size=6>**4. Feature Representations**</font> <a class="anchor" id="4"></a>

[Back to TOC](#toc)

## <font color='#BFD72F' size=5>4.1 Representation A — Previous State</font> <a class="anchor" id="4_1"></a>

[Back to TOC](#toc)

Use **`representation_a(df)`** to create the four **`BASE`** variables lagged by one hour. No feature uses its current measurement at time $t$.

In [ ]:
reps = {key: {'A': representation_a(df)} for key, df in data.items()}
for key in data:
    print(f"{names[key]} | A: {reps[key]['A'].shape[1]} features")
    display(reps[key]['A'].head())

## <font color='#BFD72F' size=5>4.2 Representation B — Stream-aware Context</font> <a class="anchor" id="4_2"></a>

[Back to TOC](#toc)

Call **`representation_b(df, cfg["features"]["windows"], y=y[key])`**. B contains the lagged state, lagged changes, rolling means and standard deviations, rain-hour counts, and lagged event rates for the configured windows. The following table lists the actual returned columns by type.

In [ ]:
windows = cfg['features']['windows']
for key, df in data.items():
    reps[key]['B'] = representation_b(df, windows, y=y[key])
feature_columns = reps['city_a']['B'].columns.tolist()
feature_groups = {
    'Previous state': [f'{c}_lag1' for c in BASE],
    'Lagged change': [f'{c}_delta1' for c in BASE],
    'Rolling mean': [f'{c}_mean{w}' for w in windows for c in BASE],
    'Rolling standard deviation': [f'{c}_std{w}' for w in windows for c in BASE],
    'Rain-hour count': [f'rain_hours_{w}' for w in windows],
    'Lagged event rate': [f'event_rate_{w}' for w in windows],
}
display(pd.DataFrame([
    {'type': group, 'columns': ', '.join(c for c in columns if c in feature_columns),
     'n_features': sum(c in feature_columns for c in columns)}
    for group, columns in feature_groups.items()
]))
for key in data:
    print(f"{names[key]} | B: {reps[key]['B'].shape[1]} features")
    display(reps[key]['B'].head())

## <font color='#BFD72F' size=5>4.3 Leakage Checks</font> <a class="anchor" id="4_3"></a>

[Back to TOC](#toc)

Check each **`*_lag1`** against `df[c].shift(1)` in both representations. For the configured six-hour window, check **`event_rate_6`** against `y.shift(1).rolling(6).mean()`. Also assert that the unlagged `precipitation` and `wind_speed` columns are absent.

In [ ]:
assert 6 in windows, 'The required event_rate_6 check needs a six-hour window.'
for key, df in data.items():
    for rep_name, X in reps[key].items():
        for c in BASE:
            pd.testing.assert_series_equal(
                X[f'{c}_lag1'], df[c].shift(1), check_names=False,
            )
        assert 'precipitation' not in X.columns
        assert 'wind_speed' not in X.columns
    np.testing.assert_allclose(
        reps[key]['B']['event_rate_6'].to_numpy(),
        y[key].shift(1).rolling(6).mean().to_numpy(),
        equal_nan=True,
    )
print('All leakage checks passed')

## <font color='#BFD72F' size=5>4.4 Shared Evaluation Mask</font> <a class="anchor" id="4_4"></a>

[Back to TOC](#toc)

Mirror the terminal mask **exactly**: `valid` uses complete **B** features and a non-missing target, and `is_warm` is based on the **original index** being below `warm_n`. Reuse this mask for both A and B, without resetting the index or redefining warm-up after filtering.

In [ ]:
valid = {}
is_warm = {}
mask_rows = []
for key, df in data.items():
    valid[key] = reps[key]['B'].notna().all(axis=1) & y[key].notna()
    is_warm[key] = pd.Series(df.index < warm_n, index=df.index)
    mask_rows.append({
        'city': names[key], 'total_rows': len(df),
        'rows_dropped_by_mask': int((~valid[key]).sum()),
        'warmup_rows_inside_valid': int((valid[key] & is_warm[key]).sum()),
        'rows_evaluated': int((valid[key] & ~is_warm[key]).sum()),
    })
display(pd.DataFrame(mask_rows))

#### **Insights** <a class="anchor" id='insights_2'></a>
*To be written by the group.*